In [1]:
import pulp
import vrplib
import numpy as np
import math
import tempfile
import os

# **Data**

In [3]:
C101_txt ="""
C101

VEHICLE
NUMBER     CAPACITY
  25         200

CUSTOMER
CUST NO.  XCOORD.   YCOORD.    DEMAND   READY TIME  DUE DATE   SERVICE   TIME

    0      40         50          0          0       1236          0
    1      45         68         10        912        967         90
    2      45         70         30        825        870         90
    3      42         66         10         65        146         90
    4      42         68         10        727        782         90
    5      42         65         10         15         67         90
    6      40         69         20        621        702         90
    7      40         66         20        170        225         90
    8      38         68         20        255        324         90
    9      38         70         10        534        605         90
   10      35         66         10        357        410         90
   11      35         69         10        448        505         90
   12      25         85         20        652        721         90
   13      22         75         30         30         92         90
   14      22         85         10        567        620         90
   15      20         80         40        384        429         90
   16      20         85         40        475        528         90
   17      18         75         20         99        148         90
   18      15         75         20        179        254         90
   19      15         80         10        278        345         90
   20      30         50         10         10         73         90
   21      30         52         20        914        965         90
   22      28         52         20        812        883         90
   23      28         55         10        732        777         90
   24      25         50         10         65        144         90
   25      25         52         40        169        224         90
   26      25         55         10        622        701         90
   27      23         52         10        261        316         90
   28      23         55         20        546        593         90
   29      20         50         10        358        405         90
   30      20         55         10        449        504         90
   31      10         35         20        200        237         90
   32      10         40         30         31        100         90
   33       8         40         40         87        158         90
   34       8         45         20        751        816         90
   35       5         35         10        283        344         90
   36       5         45         10        665        716         90
   37       2         40         20        383        434         90
   38       0         40         30        479        522         90
   39       0         45         20        567        624         90
   40      35         30         10        264        321         90
   41      35         32         10        166        235         90
   42      33         32         20         68        149         90
   43      33         35         10         16         80         90
   44      32         30         10        359        412         90
   45      30         30         10        541        600         90
   46      30         32         30        448        509         90
   47      30         35         10       1054       1127         90
   48      28         30         10        632        693         90
   49      28         35         10       1001       1066         90
   50      26         32         10        815        880         90
   51      25         30         10        725        786         90
   52      25         35         10        912        969         90
   53      44          5         20        286        347         90
   54      42         10         40        186        257         90
   55      42         15         10         95        158         90
   56      40          5         30        385        436         90
   57      40         15         40         35         87         90
   58      38          5         30        471        534         90
   59      38         15         10        651        740         90
   60      35          5         20        562        629         90
   61      50         30         10        531        610         90
   62      50         35         20        262        317         90
   63      50         40         50        171        218         90
   64      48         30         10        632        693         90
   65      48         40         10         76        129         90
   66      47         35         10        826        875         90
   67      47         40         10         12         77         90
   68      45         30         10        734        777         90
   69      45         35         10        916        969         90
   70      95         30         30        387        456         90
   71      95         35         20        293        360         90
   72      53         30         10        450        505         90
   73      92         30         10        478        551         90
   74      53         35         50        353        412         90
   75      45         65         20        997       1068         90
   76      90         35         10        203        260         90
   77      88         30         10        574        643         90
   78      88         35         20        109        170         90
   79      87         30         10        668        731         90
   80      85         25         10        769        820         90
   81      85         35         30         47        124         90
   82      75         55         20        369        420         90
   83      72         55         10        265        338         90
   84      70         58         20        458        523         90
   85      68         60         30        555        612         90
   86      66         55         10        173        238         90
   87      65         55         20         85        144         90
   88      65         60         30        645        708         90
   89      63         58         10        737        802         90
   90      60         55         10         20         84         90
   91      60         60         10        836        889         90
   92      67         85         20        368        441         90
   93      65         85         40        475        518         90
   94      65         82         10        285        336         90
   95      62         80         30        196        239         90
   96      60         80         10         95        156         90
   97      60         85         30        561        622         90
   98      58         75         20         30         84         90
   99      55         80         10        743        820         90
  100      55         85         20        647        726         90
"""

In [4]:
C101_sol = """
Route #1: 5 3 7 8 10 11 9 6 4 2 1 75
Route #2: 13 17 18 19 15 16 14 12
Route #3: 20 24 25 27 29 30 28 26 23 22 21
Route #4: 32 33 31 35 37 38 39 36 34
Route #5: 43 42 41 40 44 46 45 48 51 50 52 49 47
Route #6: 57 55 54 53 56 58 60 59
Route #7: 67 65 63 62 74 72 61 64 68 66 69
Route #8: 81 78 76 71 70 73 77 79 80
Route #9: 90 87 86 83 82 84 85 88 89 91
Route #10: 98 96 95 94 92 93 97 100 99
Cost 827.3
"""

In [5]:
# Create a temporary file to store the C101_txt content
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_file:
    temp_file.write(C101_txt)
    temp_file_path = temp_file.name

try:
    solomon_data = vrplib.read_instance(temp_file_path, instance_format='solomon')
    #display(solomon_data)
finally:
    # Clean up the temporary file
    os.remove(temp_file_path)


In [9]:
num_locations = len(solomon_data['node_coord'])
num_vehicles = solomon_data['vehicles']
capacity = float(solomon_data['capacity'])
cust_demand = solomon_data['demand'].tolist()
cust_demand = [float(d) for d in cust_demand]
avail_time = solomon_data['time_window'][:, 0].tolist()
avail_time = [float(d) for d in avail_time]
due_date = solomon_data['time_window'][:, 1].tolist()
due_date = [float(d) for d in due_date]
serve_time= solomon_data['service_time'].tolist()
serve_time = [float(d) for d in serve_time]

print(f"Number of locations : {num_locations}")
print(f"Number of vehicles : {num_vehicles}")
print(f"Vehicle capacity (q): {capacity}")
print(f"Demand : {cust_demand}")
print(f"Ready time: {avail_time}")
print(f"Due date (b): {due_date}")
print(f"Service time (s): {serve_time}")

# Extract coordinates for distance calculation
coords = solomon_data['node_coord']

# Initialize the cost matrix
travel_cost = np.zeros((num_locations, num_locations))

# Calculate Euclidean distances
for i in range(num_locations):
    for j in range(num_locations):
        if i == j:
            travel_cost[i, j] = 0
        else:
            travel_cost[i, j] = math.dist(coords[i], coords[j])

# Convert to list of lists (if DIDPPY expects this format) and round to nearest integer as typically done in VRP
travel_cost = [[float(val) for val in row] for row in travel_cost]

print(f"\nTravel cost matrix (c):\n{np.array(travel_cost)}")

Number of locations : 101
Number of vehicles : 25
Vehicle capacity (q): 200.0
Demand : [0.0, 10.0, 30.0, 10.0, 10.0, 10.0, 20.0, 20.0, 20.0, 10.0, 10.0, 10.0, 20.0, 30.0, 10.0, 40.0, 40.0, 20.0, 20.0, 10.0, 10.0, 20.0, 20.0, 10.0, 10.0, 40.0, 10.0, 10.0, 20.0, 10.0, 10.0, 20.0, 30.0, 40.0, 20.0, 10.0, 10.0, 20.0, 30.0, 20.0, 10.0, 10.0, 20.0, 10.0, 10.0, 10.0, 30.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 20.0, 40.0, 10.0, 30.0, 40.0, 30.0, 10.0, 20.0, 10.0, 20.0, 50.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 30.0, 20.0, 10.0, 10.0, 50.0, 20.0, 10.0, 10.0, 20.0, 10.0, 10.0, 30.0, 20.0, 10.0, 20.0, 30.0, 10.0, 20.0, 30.0, 10.0, 10.0, 10.0, 20.0, 40.0, 10.0, 30.0, 10.0, 30.0, 20.0, 10.0, 20.0]
Ready time: [0.0, 912.0, 825.0, 65.0, 727.0, 15.0, 621.0, 170.0, 255.0, 534.0, 357.0, 448.0, 652.0, 30.0, 567.0, 384.0, 475.0, 99.0, 179.0, 278.0, 10.0, 914.0, 812.0, 732.0, 65.0, 169.0, 622.0, 261.0, 546.0, 358.0, 449.0, 200.0, 31.0, 87.0, 751.0, 283.0, 665.0, 383.0, 479.0, 567.0, 264.0, 166.0, 68.0, 16.0

In [10]:
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as temp_sol_file:
    temp_sol_file.write(C101_sol)
    temp_sol_file_path = temp_sol_file.name

try:
    solomon_solution = vrplib.read_solution(temp_sol_file_path)
    print("Solution Data:")
    display(solomon_solution)
finally:
    os.remove(temp_sol_file_path)

Solution Data:


{'routes': [[5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1, 75],
  [13, 17, 18, 19, 15, 16, 14, 12],
  [20, 24, 25, 27, 29, 30, 28, 26, 23, 22, 21],
  [32, 33, 31, 35, 37, 38, 39, 36, 34],
  [43, 42, 41, 40, 44, 46, 45, 48, 51, 50, 52, 49, 47],
  [57, 55, 54, 53, 56, 58, 60, 59],
  [67, 65, 63, 62, 74, 72, 61, 64, 68, 66, 69],
  [81, 78, 76, 71, 70, 73, 77, 79, 80],
  [90, 87, 86, 83, 82, 84, 85, 88, 89, 91],
  [98, 96, 95, 94, 92, 93, 97, 100, 99]],
 'cost': 827.3}

# **LP relax**

In [ ]:
def create_cvrptw_relaxed_model(num_locations, num_vehicles, capacity, 
                                cust_demand, travel_cost, 
                                serve_time, ready_time, due_date):
    """
    Creates the CVRPTW Linear Programming Relaxation model.
    Based strictly on Cordeau (2002) formulation (Eq 7.1 - 7.11).
    
    Notation:
      N: Set of customers {1, ..., n}
      V: Set of nodes {0, 1, ..., n, n+1} (0 is start depot, n+1 is end depot)
      A: Set of Arcs
      Delta+(i): Set of nodes j such that (i,j) in A
      Delta-(i): Set of nodes j such that (j,i) in A
    """
    
    # --- 0. Data Augmentation (Add Dummy Depot n+1) ---
    # The end depot (n+1) is a copy of the start depot (0) physically, 
    # but logically distinct for flow conservation.
    
    # New indices
    n_customers = num_locations - 1 # Original customers (indices 1 to n)
    StartDepot = 0
    EndDepot = num_locations        # This is the new index 'n+1'
    
    # Duplicate Data for EndDepot
    # Demand is 0
    aug_demand = cust_demand + [0.0]
    # Service time is 0
    aug_serve = serve_time + [0.0]
    # Time window is same as StartDepot
    aug_ready = ready_time + [ready_time[0]]
    aug_due = due_date + [due_date[0]]
    
    # Expand Travel Cost Matrix (Add row and col for EndDepot)
    # Cost to EndDepot = Cost to StartDepot
    aug_cost = [row[:] + [row[0]] for row in travel_cost] 
    # Cost from EndDepot = Cost from StartDepot (though no flow leaves EndDepot)
    aug_cost.append(aug_cost[0][:]) 

    # --- 1. Sets (actual number of customers, not indices) ---
    # K: Set of vehicles
    K = range(num_vehicles)
    # N: Set of customers {1, ..., n}
    N = range(1, num_locations)
    # V: Set of all nodes {0, ..., n, n+1}
    V = range(num_locations + 1)
    
    # A: Arc Set 
    # Logic: 
    # - Leave StartDepot (0 -> j)
    # - Inter-customer (i -> j)
    # - Enter EndDepot (i -> n+1)
    # - NO arcs entering StartDepot
    # - NO arcs leaving EndDepot
    A = []
    for i in V:
        for j in V:
            if i == j: continue
            if i == EndDepot: continue   # Nothing leaves EndDepot
            if j == StartDepot: continue # Nothing enters StartDepot
            A.append((i,j))
            
    # Helper to get Delta+(i) (Outgoing neighbors)
    def delta_plus(i):
        return [j for (u, j) in A if u == i]

    # Helper to get Delta-(i) (Incoming neighbors)
    def delta_minus(i):
        return [j for (j, v) in A if v == i]

    # Calculate Big-M for Linearized Time Constraint (7.6a)
    # M_ij = max(b_i + s_i + t_ij - a_j, 0)
    big_m = {}
    for (i, j) in A:
        val = aug_due[i] + aug_serve[i] + aug_cost[i][j] - aug_ready[j]
        big_m[(i,j)] = max(val, 0)

    # --- 2. Initialize Model ---
    mdl = pulp.LpProblem("CVRPTW_Relaxed_Cordeau", pulp.LpMinimize)

    # --- 3. Create Variables ---
    
    # x_ijk: Flow variables (Relaxed to Continuous [0,1])
    # Equal to 1 if arc (i,j) is used by vehicle k
    x = {}
    for k in K:
        for (i, j) in A:
            x[(k, i, j)] = pulp.LpVariable(f"x_{k}_{i}_{j}", 0, 1, pulp.LpContinuous)

    # w_ik: Start of Service Time variables (Continuous)
    # Specifying the start of service at node i when serviced by vehicle k
    w = {}
    for k in K:
        for i in V:
            # Bounds from (7.8): E <= w_ik <= L
            # We use specific node bounds [a_i, b_i]
            w[(k, i)] = pulp.LpVariable(f"w_{k}_{i}", 0, 10**6, pulp.LpContinuous)

    # --- 4. Objective Function (7.1) ---
    # min sum(c_ij * x_ijk)
    mdl += pulp.lpSum(aug_cost[i][j] * x[(k, i, j)] 
                      for k in K 
                      for (i, j) in A), "Minimize_Total_Cost"

    # --- 5. Constraints ---

    # (7.2) Assignment: Each customer visited exactly once
    # sum(x_ijk) over k, j in Delta+(i) = 1  for all i in N
    for i in N:
        mdl += pulp.lpSum(x[(k, i, j)] 
                          for k in K 
                          for j in delta_plus(i)) == 1, f"Eq_7_2_Assign_{i}"

    # (7.3) Depot Departure
    # sum(x_0jk) over j in Delta+(0) = 1 for all k in K
    for k in K:
        mdl += pulp.lpSum(x[(k, StartDepot, j)] 
                          for j in delta_plus(StartDepot)) == 1, f"Eq_7_3_DepotOut_{k}"

    # (7.4) Flow Conservation
    # sum(x_ijk) over i in Delta-(j) - sum(x_jik) over i in Delta+(j) = 0
    # for all k in K, j in N
    for k in K:
        for j in N:
            inflow = pulp.lpSum(x[(k, i, j)] for i in delta_minus(j))
            outflow = pulp.lpSum(x[(k, j, i)] for i in delta_plus(j))
            mdl += inflow - outflow == 0, f"Eq_7_4_FlowBal_{k}_{j}"

    # (7.5) Depot Arrival (Returning to EndDepot)
    # sum(x_i,n+1,k) = 1.
    # sum(x_i,EndDepot,k) over i in Delta-(EndDepot) = 1 for all k in K
    for k in K:
        mdl += pulp.lpSum(x[(k, i, EndDepot)] 
                          for i in delta_minus(EndDepot)) == 1, f"Eq_7_5_DepotIn_{k}"

    # (7.9) Capacity Constraints
    # sum(d_i * sum(x_ijk)) <= C for all k in K
    for k in K:
        # Note: Inner sum is over j in Delta+(i) to check if i is visited by k
        load_sum = pulp.lpSum(aug_demand[i] * pulp.lpSum(x[(k, i, j)] for j in delta_plus(i))
                              for i in N)
        mdl += load_sum <= capacity, f"Eq_7_9_Capacity_{k}"

    # (7.6a) Linearized Time Propagation (Big-M)
    # w_ik + s_i + t_ij - w_jk <= (1 - x_ijk) * M_ij
    # for all k in K, (i,j) in A   
    for k in K:
        for (i, j) in A:
            M = 10**6 #big_m[(i,j)]
            if M > 0:
                # w_i + s_i + t_ij - w_j <= M(1-x)
                lhs = w[(k, i)] + aug_serve[i] + aug_cost[i][j] - w[(k, j)]
                rhs = M * (1 - x[(k, i, j)])
                mdl += lhs <= rhs, f"Eq_7_6a_TimeProp_{k}_{i}_{j}"

    # (7.7) Time Windows Linked to Visits
    # a_i * sum(x) <= w_ik <= b_i * sum(x)
    # This forces w_ik = 0 if customer i is not visited by vehicle k
    for k in K:
        for i in N:
            # sum(x_ijk) over j in Delta+(i)
            visit_var = pulp.lpSum(x[(k, i, j)] for j in delta_plus(i))
            
            mdl += w[(k, i)] >= aug_ready[i] * visit_var, f"Eq_7_7_TW_Start_{k}_{i}"
            mdl += w[(k, i)] <= aug_due[i] * visit_var, f"Eq_7_7_TW_End_{k}_{i}"

    return mdl#, big_m

# **Execution**

In [33]:
# --- Build the Model ---
print("\n--- Building CVRPTW Relaxation Model (Pulp) ---")

# Note: We map your specific variable names to the function arguments
# q -> capacity
# avail_time -> ready_time
mdl = create_cvrptw_relaxed_model(
    num_locations=num_locations,
    num_vehicles=num_vehicles,
    capacity=capacity, 
    cust_demand=cust_demand,
    travel_cost=travel_cost,
    serve_time=serve_time,
    ready_time=avail_time,
    due_date=due_date
)


--- Building CVRPTW Relaxation Model (Pulp) ---


In [ ]:
# --- Solve ---
print("--- Solving with CPLEX ---")
# Time limit set to 60 seconds. msg=True to see the log.
solver = pulp.CPLEX_CMD(timeLimit=60, msg=True, threads= 2)

try:
    mdl.solve(solver)
except Exception as e:
    print(f"CPLEX failed ({e}), trying default CBC solver...")
    mdl.solve()

In [31]:
# --- Output Results ---
if mdl.status == pulp.LpStatusOptimal or mdl.status == pulp.LpStatusNotSolved:
    print("\n" + "="*40)
    print(f"LOWER BOUND (Objective): {pulp.value(mdl.objective):.4f}")
    print(f"Solve Status: {pulp.LpStatus[mdl.status]}")
    print("="*40)
    
    # Optional: Check non-zero flows
    print("\nSignificant Fractional Flows (> 0.01):")
    var_dict = mdl.variablesDict()
    count = 0
    
    # Iterate through the augmented set of nodes used inside the model
    # The model uses indices 0 to num_locations (where num_locations is the dummy depot)
    internal_n = num_locations + 1 
    
    for k in range(num_vehicles):
        for i in range(internal_n):
            for j in range(internal_n):
                if i != j:
                    var_name = f'x_{k}_{i}_{j}'
                    if var_name in var_dict:
                        val = var_dict[var_name].varValue
                        if val and val >= 0:
                            # If j equals the original num_locations, it's the dummy end depot
                            dest_name = "EndDepot" if j == num_locations else f"{j}"
                            print(f"Veh {k} | {i} -> {dest_name} : {val:.2f}")
                            count += 1
                            if count > 10: break 
        if count > 10: break
else:
    print("\nNo solution found")
    print(f"Status: {pulp.LpStatus[mdl.status]}")


LOWER BOUND (Objective): 308.8520
Solve Status: Optimal

Significant Fractional Flows (> 0.01):
Veh 0 | 0 -> EndDepot : 1.00
Veh 0 | 1 -> 2 : 0.00
Veh 0 | 2 -> 1 : 0.00
Veh 0 | 3 -> 7 : 0.00
Veh 0 | 4 -> 6 : 0.00
Veh 0 | 5 -> 75 : 0.00
Veh 0 | 6 -> 4 : 0.00
Veh 0 | 7 -> 3 : 0.00
Veh 0 | 8 -> 9 : 0.00
Veh 0 | 9 -> 8 : 0.00
Veh 0 | 10 -> 11 : 0.00
Veh 0 | 11 -> 10 : 0.00
Veh 0 | 20 -> 22 : 1.00
Veh 0 | 21 -> 20 : 1.00
Veh 0 | 22 -> 21 : 1.00
Veh 0 | 23 -> 26 : 0.00
Veh 0 | 24 -> 25 : 0.00
Veh 0 | 25 -> 24 : 0.00
Veh 0 | 26 -> 23 : 0.00
Veh 0 | 27 -> 29 : 0.00
Veh 0 | 28 -> 30 : 0.00
Veh 0 | 29 -> 27 : 0.00
Veh 0 | 30 -> 28 : 0.00
Veh 0 | 31 -> 35 : 0.00
Veh 0 | 34 -> 36 : 0.00
Veh 0 | 35 -> 31 : 0.00
Veh 0 | 36 -> 34 : 0.00
Veh 0 | 37 -> 39 : 1.00
Veh 0 | 38 -> 37 : 1.00
Veh 0 | 39 -> 38 : 1.00
Veh 0 | 40 -> 41 : 0.00
Veh 0 | 41 -> 40 : 0.00
Veh 0 | 43 -> 47 : 0.00
Veh 0 | 45 -> 48 : 1.00
Veh 0 | 46 -> 45 : 1.00
Veh 0 | 47 -> 43 : 0.00
Veh 0 | 48 -> 46 : 1.00
Veh 0 | 49 -> 52 : 0.00
Veh